# EdgeForge - Part 1: Knowledge Distillation (CIFAR-100)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sudoHasaan/edgeforge/blob/main/notebooks/01_knowledge_distillation.ipynb)

Pipeline:
1. Train a **ResNet-50 teacher** on CIFAR-100
2. Train **baseline students** (MobileNetV2, SqueezeNet, EfficientNet-Lite0) with cross-entropy only
3. **Distil** the teacher into the same students: soft targets with temperature + hard labels + intermediate feature hints
4. Unified **PyTorch inference** for all models
5. **ONNX export** + **ONNX Runtime CPU inference**
6. **Benchmarking** (accuracy, size, CPU latency) and report figures

> **Before running:** `Runtime -> Change runtime type -> T4 GPU`.
> Every training step resumes automatically if Colab disconnects - just re-run the cell.

## 1. Setup

In [ ]:
import os
REPO_URL = "https://github.com/sudoHasaan/edgeforge.git"
REPO_DIR = "/content/edgeforge"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Persist outputs to Google Drive (recommended)

`checkpoints/`, `results/`, `onnx/` and `data/` (dataset cache) are linked to `MyDrive/edgeforge_runs/`, so trained models
survive a disconnect and training resumes from the last finished epoch.

In [ ]:
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/edgeforge_runs"

if USE_DRIVE:
    import pathlib, shutil
    from google.colab import drive
    drive.mount("/content/drive")
    for d in ["checkpoints", "results", "onnx", "data"]:
        target = pathlib.Path(DRIVE_DIR) / d
        target.mkdir(parents=True, exist_ok=True)
        local = pathlib.Path(REPO_DIR) / d
        if local.is_symlink():
            continue
        if local.exists():
            shutil.copytree(local, target, dirs_exist_ok=True)
            shutil.rmtree(local)
        local.symlink_to(target, target_is_directory=True)
    print("Outputs are saved to", DRIVE_DIR)

## 3. Configuration

The per-epoch time is printed during training. To check quickly that everything works, set the epochs to `1` first.

In [ ]:
TEACHER_EPOCHS = 30
STUDENT_EPOCHS = 30
BATCH_SIZE     = 128

# Distillation hyper-parameters
ALPHA       = 0.9   # weight of soft-target loss; (1 - ALPHA) weights the hard-label CE loss
TEMPERATURE = 4.0   # softmax temperature
BETA        = 1.0   # weight of intermediate feature distillation (0 disables it)

## 4. Train the ResNet-50 teacher

In [ ]:
!python scripts/train_teacher.py --epochs {TEACHER_EPOCHS} --batch-size {BATCH_SIZE}

## 5. Baseline students (no distillation)

In [ ]:
!python scripts/train_baseline.py --students all --epochs {STUDENT_EPOCHS} --batch-size {BATCH_SIZE}

## 6. Knowledge distillation

Loss = `(1 - alpha) * CE(student, y) + alpha * T^2 * KL(teacher_T || student_T) + beta * FeatureHint(mid, final)`

In [ ]:
!python scripts/train_kd.py --students all --epochs {STUDENT_EPOCHS} --batch-size {BATCH_SIZE}     --alpha {ALPHA} --temperature {TEMPERATURE} --beta {BETA}

### 6b. (Optional) Ablation: logit-only KD

Trains the students again with feature distillation turned off (`beta = 0`), to show what the feature hints add.
Outputs go to `checkpoints/ablation_logit_kd/`.

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    !python scripts/train_kd.py --students all --epochs {STUDENT_EPOCHS} --batch-size {BATCH_SIZE}         --alpha {ALPHA} --temperature {TEMPERATURE} --beta 0 --out-dir checkpoints/ablation_logit_kd

## 7. Unified PyTorch inference

In [ ]:
# Test-set accuracy of the teacher and every student
!python scripts/infer.py --model resnet50 all --variant all --eval --save-csv results/kd_eval.csv

In [ ]:
# Predict random test images with the distilled students
!python scripts/infer.py --model all --variant distilled --samples 5

In [ ]:
# Visual check: predictions of each distilled student on 8 test images
import os, sys, torch, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, REPO_DIR)
from src.common.data import load_raw
from src.common.inference import Predictor
from src.common.paths import ckpt_path, STUDENT_NAMES

x, y, classes = load_raw("cifar100", train=False)
idx = np.random.default_rng(1).choice(len(x), 8, replace=False)
imgs, labels = x[idx].permute(0, 2, 3, 1).numpy(), y[idx].numpy()

fig, axes = plt.subplots(len(STUDENT_NAMES), 8, figsize=(16, 2.4 * len(STUDENT_NAMES)))
for r, name in enumerate(STUDENT_NAMES):
    preds = Predictor(ckpt_path(name, "distilled"), "cuda" if torch.cuda.is_available() else "cpu").predict(imgs, topk=1)
    for c in range(8):
        ax = axes[r, c]
        ax.imshow(imgs[c]); ax.set_xticks([]); ax.set_yticks([])
        label, p = preds[c][0]
        ok = label == classes[labels[c]]
        ax.set_title(f"{label}
{p:.0%}", fontsize=8, color="#1f1f1d" if ok else "#c0392b")
        if c == 0:
            ax.set_ylabel(name, fontsize=9)
fig.suptitle("Distilled students - predictions (red = wrong)", fontweight="bold")
plt.tight_layout()
os.makedirs("results/plots", exist_ok=True)
plt.savefig("results/plots/kd_sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Export to ONNX

In [ ]:
!python scripts/export_onnx.py --model resnet50 all --variant all

## 9. ONNX Runtime CPU inference

In [ ]:
!python scripts/onnx_infer.py --model all --variant all --eval --latency

In [ ]:
!python scripts/onnx_infer.py --model all --variant distilled --samples 5

## 10. Benchmark (accuracy, size, CPU latency: PyTorch vs ONNX Runtime)

In [ ]:
!python scripts/benchmark_kd.py --runs 100

import pandas as pd
bench = pd.read_csv("results/kd_benchmark.csv")
bench

## 11. Report figures

In [ ]:
!python scripts/make_plots_kd.py

from IPython.display import Image, display
for f in ["teacher_training", "kd_training_curves", "kd_accuracy_comparison",
          "kd_accuracy_vs_latency", "kd_size_latency"]:
    display(Image(f"results/plots/{f}.png"))

## 12. Download all outputs

Creates `edgeforge_q1_outputs.zip` with `checkpoints/`, `onnx/` and `results/` (folder layout matches the repo).

In [ ]:
!cd {REPO_DIR} && rm -f /content/edgeforge_q1_outputs.zip &&     zip -qr /content/edgeforge_q1_outputs.zip checkpoints onnx results -x "*_last.pth" "*.gitkeep"
!ls -lh /content/edgeforge_q1_outputs.zip
if USE_DRIVE:
    !cp /content/edgeforge_q1_outputs.zip {DRIVE_DIR}/
from google.colab import files
files.download("/content/edgeforge_q1_outputs.zip")